# Quantum ML-based Classification of Leukoplakia with Dysplasia (Histopathological Images)
Implementation of the paper's methodology: **C-GAN augmentation → CML (CNN) and QML (quanvolutional NN) → 4 baselines → 6 metrics (mean ± SD over 5 runs) → Grad-CAM**.

**Classes (4):** Leukoplakia with dysplasia · Leukoplakia without dysplasia · OSCC · Normal

**Expected folder layout** (Mendeley datasets [11] and [26], any sub-folder nesting is fine; the class is read from the folder name):
```
DATA_ROOT/
  Dataset1/With_Dysplasia/*.png     Dataset1/Without_Dysplasia/*.png     Dataset1/OSCC/*.png
  Dataset2/OSCC/*.jpg               Dataset2/Normal/*.jpg
```
**Assumptions where the paper is vague/inconsistent** (all are switches in the Config cell):
- Paper's C-GAN works on 28×28×1, CML on 150×150×3, QML gives 20×20 quantum images (→ QML input = 40×40 gray, 2×2 patches, stride 2). Synthetic 28×28 images are up-sampled for the classifiers.
- Latent dim: Table 3 says 64, Table 4 says 100 → 100 used. Softmax has **4** outputs (paper text says 2, which is a typo for 4 classes).
- ResNet-151 does not exist in Keras → `ResNet152` is used.
- `LEAK_FREE=True`: split real images first and train the GAN on the train split only (synthetic images go only to train). Set `False` for the paper's literal order (augment everything, then 70/20/10 split), but then test accuracy is inflated by leakage.

In [ ]:
!pip install -q tensorflow pennylane scikit-learn matplotlib seaborn pillow

In [ ]:
import os, re, json, time, random, math
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, matthews_corrcoef
import pennylane as qml

# ----------------------------- CONFIG -----------------------------
QUICK_TEST   = False                      # True = tiny run just to check the code works
DATA_ROOT    = Path(os.environ.get("DATA_ROOT", "/content/data"))
OUT_DIR      = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)
SEED         = 42
LEAK_FREE    = True

CLASS_NAMES  = ["Leukoplakia_with_dysplasia", "Leukoplakia_without_dysplasia", "OSCC", "Normal"]
NUM_CLASSES  = len(CLASS_NAMES)

# preprocessing
CML_SIZE     = 150          # CML + baselines: 150x150x3
GAN_SIZE     = 28           # C-GAN: 28x28x1
QML_SIZE     = 40           # 40x40 gray -> 2x2 patches -> 20x20x4 quantum image
SPLIT        = (0.70, 0.20, 0.10)      # train / val / test

# C-GAN (Table 3 / 4)
GAN_EPOCHS, GAN_LR, GAN_BATCH, GAN_BUFFER = 500, 1e-4, 64, 1000
LATENT_DIM, GAN_BETA1, GAN_DROPOUT, GAN_VAL_FREQ = 100, 0.5, 0.5, 100
SYN_TARGET   = {"Leukoplakia_with_dysplasia": 4237, "Leukoplakia_without_dysplasia": 1608,
                "OSCC": 5379, "Normal": 765}      # synthetic counts from Table 5

# classifiers
EPOCHS, BATCH, LR, N_RUNS = 500, 32, 1e-4, 5
DROPOUT_CML, DROPOUT_QML  = 0.05, 0.0001
EARLY_STOP, PATIENCE      = True, 20        # paper trains 500 epochs; early stopping saves time
BASELINE_WEIGHTS          = None            # None = from scratch, or "imagenet"
N_QUBITS, QUANV_LAYERS    = 4, 1

if QUICK_TEST:
    GAN_EPOCHS, EPOCHS, N_RUNS, GAN_VAL_FREQ = 2, 2, 2, 1
    SYN_TARGET = {k: 20 for k in SYN_TARGET}

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TF", tf.__version__, "| PennyLane", qml.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Data loading & preprocessing
Steps: (1) walk dataset folders, (2) infer class from folder name (merging OSCC of both datasets), (3) convert PNG/JPG to RGB, (4) resize to 150×150, kept as `uint8` to save RAM (the models rescale to [0,1] internally), (5) stratified 70/20/10 split.

In [ ]:
def folder_to_class(path: Path):
    for part in reversed(path.parent.parts):                 # nearest parent folder first
        p = re.sub(r"[^a-z]", "", part.lower())
        if "without" in p and "dysplasia" in p: return "Leukoplakia_without_dysplasia"
        if "dysplasia" in p:                    return "Leukoplakia_with_dysplasia"
        if "oscc" in p or "carcinoma" in p or "cancer" in p: return "OSCC"
        if "normal" in p:                       return "Normal"
    return None

def load_real_images(root: Path, size=CML_SIZE):
    X, y, skipped = [], [], 0
    for f in sorted(root.rglob("*")):
        if f.suffix.lower() not in (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"): continue
        c = folder_to_class(f)
        if c is None: skipped += 1; continue
        img = Image.open(f).convert("RGB").resize((size, size), Image.BILINEAR)
        X.append(np.asarray(img, dtype=np.uint8)); y.append(CLASS_NAMES.index(c))
    print(f"loaded {len(X)} images, skipped (unknown class folder): {skipped}")
    assert len(X) > 0, f"No images found under {root}. Check DATA_ROOT and the folder names."
    return np.stack(X), np.array(y)

if QUICK_TEST and not DATA_ROOT.exists():          # fake data, only for checking the pipeline
    rng = np.random.default_rng(0)
    for c in CLASS_NAMES:
        (DATA_ROOT/"fake"/c).mkdir(parents=True, exist_ok=True)
        for i in range(40):
            Image.fromarray(rng.integers(0,255,(200,200,3),dtype=np.uint8)).save(DATA_ROOT/"fake"/c/f"{i}.png")

X_real, y_real = load_real_images(DATA_ROOT)
print(X_real.shape, dict(zip(CLASS_NAMES, np.bincount(y_real, minlength=NUM_CLASSES))))

In [ ]:
def strat_split(X, y, fr=SPLIT, seed=SEED):
    tr, va, te = fr
    Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=1-tr, stratify=y, random_state=seed)
    Xva, Xte, yva, yte  = train_test_split(Xtmp, ytmp, test_size=te/(va+te), stratify=ytmp, random_state=seed)
    return (Xtr, ytr), (Xva, yva), (Xte, yte)

def to_gray_size(X, size, chunk=512):
    # uint8 RGB (N,H,W,3) -> uint8 gray (N,size,size,1)
    out = []
    for i in range(0, len(X), chunk):
        g = tf.image.rgb_to_grayscale(tf.cast(X[i:i+chunk], tf.float32))
        out.append(tf.cast(tf.round(tf.image.resize(g, (size, size))), tf.uint8).numpy())
    return np.concatenate(out)

def gray_to_rgb_size(X, size, chunk=512):
    # uint8 gray (N,h,w,1) -> uint8 RGB (N,size,size,3)
    out = []
    for i in range(0, len(X), chunk):
        r = tf.image.resize(tf.cast(X[i:i+chunk], tf.float32), (size, size))
        out.append(tf.cast(tf.round(tf.repeat(r, 3, axis=-1)), tf.uint8).numpy())
    return np.concatenate(out)

if LEAK_FREE:
    (Xtr, ytr), (Xva, yva), (Xte, yte) = strat_split(X_real, y_real)
    gan_X, gan_y = Xtr, ytr
else:
    gan_X, gan_y = X_real, y_real
print("images used to train the C-GAN:", len(gan_X))

## 2. Phase I – Conditional GAN (Section 3.1.2, Tables 3 & 4)
Generator: noise(100) + class embedding → project/reshape 7×7×256 → TConv 14×14×128 → TConv 28×28×64 → TConv 28×28×1 (tanh), each with BN + LeakyReLU, 5×5 kernels, no bias.
Discriminator: 28×28×1 (+ label plane) → Dropout 0.5 → Conv 64 → Conv 128 → Conv 128 → Conv 256 → Conv 1×1×1, LeakyReLU/BN. BCE loss, Adam(1e-4, β1=0.5).

In [ ]:
def build_generator():
    z   = layers.Input((LATENT_DIM,)); lab = layers.Input((1,), dtype="int32")
    e   = layers.Flatten()(layers.Embedding(NUM_CLASSES, 50)(lab))
    x   = layers.Concatenate()([z, e])
    x   = layers.Dense(7*7*256, use_bias=False)(x); x = layers.BatchNormalization()(x); x = layers.LeakyReLU(0.2)(x)
    x   = layers.Reshape((7,7,256))(x)
    x   = layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False)(x)   # 14x14x128
    x   = layers.BatchNormalization()(x); x = layers.LeakyReLU(0.2)(x)
    x   = layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False)(x)    # 28x28x64
    x   = layers.BatchNormalization()(x); x = layers.LeakyReLU(0.2)(x)
    out = layers.Conv2DTranspose(1, 5, strides=1, padding="same", activation="tanh", use_bias=False)(x)  # 28x28x1
    return keras.Model([z, lab], out, name="generator")

def build_discriminator():
    img = layers.Input((GAN_SIZE, GAN_SIZE, 1)); lab = layers.Input((1,), dtype="int32")
    e   = layers.Dense(GAN_SIZE*GAN_SIZE)(layers.Flatten()(layers.Embedding(NUM_CLASSES, 50)(lab)))
    e   = layers.Reshape((GAN_SIZE, GAN_SIZE, 1))(e)
    x   = layers.Concatenate()([img, e])
    x   = layers.Dropout(GAN_DROPOUT)(x)
    x   = layers.Conv2D(64, 5, strides=1, padding="same")(x);  x = layers.LeakyReLU(0.2)(x)                       # 28x28x64
    x   = layers.Conv2D(128, 5, strides=2, padding="same")(x); x = layers.BatchNormalization()(x); x = layers.LeakyReLU(0.2)(x)  # 14x14x128
    x   = layers.Conv2D(128, 5, strides=2, padding="same")(x); x = layers.LeakyReLU(0.2)(x)                       # 7x7x128
    x   = layers.Conv2D(256, 3, strides=2, padding="valid")(x); x = layers.BatchNormalization()(x); x = layers.LeakyReLU(0.2)(x)  # 3x3x256
    out = layers.Conv2D(1, 3, padding="valid")(x)                                                                 # 1x1x1 (logit)
    return keras.Model([img, lab], layers.Flatten()(out), name="discriminator")

G, D = build_generator(), build_discriminator()
bce  = keras.losses.BinaryCrossentropy(from_logits=True)
g_opt = keras.optimizers.Adam(GAN_LR, beta_1=GAN_BETA1); d_opt = keras.optimizers.Adam(GAN_LR, beta_1=GAN_BETA1)

@tf.function
def gan_step(real, lab):
    n = tf.shape(real)[0]; z = tf.random.normal((n, LATENT_DIM))
    with tf.GradientTape() as gt, tf.GradientTape() as dt:
        fake  = G([z, lab], training=True)
        d_r, d_f = D([real, lab], training=True), D([fake, lab], training=True)
        g_loss = bce(tf.ones_like(d_f), d_f)                                   # -mean(log D(G(z)))   Eq.34
        d_loss = bce(tf.ones_like(d_r), d_r) + bce(tf.zeros_like(d_f), d_f)    # discriminator loss
    g_opt.apply_gradients(zip(gt.gradient(g_loss, G.trainable_variables), G.trainable_variables))
    d_opt.apply_gradients(zip(dt.gradient(d_loss, D.trainable_variables), D.trainable_variables))
    d_acc = 0.5*(tf.reduce_mean(tf.cast(tf.sigmoid(d_r) > .5, tf.float32)) + tf.reduce_mean(tf.cast(tf.sigmoid(d_f) < .5, tf.float32)))
    return g_loss, d_loss, d_acc

# GAN preprocessing: gray 28x28, scaled to [-1, 1]
gan_imgs = (to_gray_size(gan_X, GAN_SIZE).astype("float32") - 127.5) / 127.5
ds = (tf.data.Dataset.from_tensor_slices((gan_imgs, gan_y.astype("int32").reshape(-1,1)))
      .shuffle(GAN_BUFFER, seed=SEED).batch(GAN_BATCH, drop_remainder=False).prefetch(2))

def show_samples(ep):
    z = tf.random.normal((NUM_CLASSES*6, LATENT_DIM)); l = tf.constant(np.repeat(np.arange(NUM_CLASSES), 6).reshape(-1,1), tf.int32)
    s = G([z, l], training=False).numpy()[...,0]
    fig, ax = plt.subplots(NUM_CLASSES, 6, figsize=(8, 1.4*NUM_CLASSES))
    for i, a in enumerate(ax.ravel()): a.imshow(s[i], cmap="gray"); a.axis("off")
    for r in range(NUM_CLASSES): ax[r,0].set_title(CLASS_NAMES[r], fontsize=7, loc="left")
    fig.suptitle(f"C-GAN samples - epoch {ep}"); plt.show()

hist = []
for ep in range(1, GAN_EPOCHS+1):
    m = np.mean([[float(v) for v in gan_step(x, y)] for x, y in ds], axis=0)
    hist.append(m)
    if ep % GAN_VAL_FREQ == 0 or ep == 1:
        print(f"epoch {ep}/{GAN_EPOCHS}  G-loss {m[0]:.3f}  D-loss {m[1]:.3f}  D-acc {m[2]:.3f}"); show_samples(ep)
hist = np.array(hist)
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, t, i in zip(ax, ["Generator loss", "Discriminator loss", "Discriminator accuracy"], range(3)): a.plot(hist[:, i]); a.set_title(t)
plt.show()
G.save(OUT_DIR/"cgan_generator.keras")

## 3. Synthetic image generation (Table 5) and final datasets

In [ ]:
def generate_synthetic(counts: dict, bs=256):
    Xs, ys = [], []
    for cname, n in counts.items():
        c = CLASS_NAMES.index(cname)
        for i in range(0, n, bs):
            b = min(bs, n-i)
            g = G([tf.random.normal((b, LATENT_DIM)), tf.fill((b,1), tf.constant(c, tf.int32))], training=False).numpy()
            Xs.append(np.clip(np.round(g*127.5+127.5), 0, 255).astype(np.uint8)); ys.append(np.full(b, c))
    return np.concatenate(Xs), np.concatenate(ys)

Xs28, ys = generate_synthetic(SYN_TARGET)
Xs150 = gray_to_rgb_size(Xs28, CML_SIZE)          # synthetic 28x28 gray -> 150x150x3 for CML/baselines
print("synthetic:", Xs150.shape, dict(zip(CLASS_NAMES, np.bincount(ys, minlength=NUM_CLASSES))))

if LEAK_FREE:   # synthetic images only enter the TRAIN split
    Xtr = np.concatenate([Xtr, Xs150]); ytr = np.concatenate([ytr, ys])
else:           # paper order: real + synthetic, then split 70/20/10
    (Xtr, ytr), (Xva, yva), (Xte, yte) = strat_split(np.concatenate([X_real, Xs150]), np.concatenate([y_real, ys]))

p = np.random.permutation(len(Xtr)); Xtr, ytr = Xtr[p], ytr[p]
for n, (a, b) in {"train": (Xtr, ytr), "val": (Xva, yva), "test": (Xte, yte)}.items():
    print(n, a.shape, np.bincount(b, minlength=NUM_CLASSES))

## 4. Quanvolutional layer (Section 3.3.2, Eq. 16, Fig. 5)
Each 2×2 patch of the 40×40 gray image → angle-encoded into a **4-qubit** circuit (`RY(π·x)` on each qubit) → fixed random quantum layer (`RandomLayers`, the quantum "convolution kernel") → ⟨Z⟩ on every qubit = 4 output channels. 40×40 with stride 2 → **20×20×4 quantum image**. The circuit is fixed (not trained), so it's computed once and cached.

In [ ]:
dev = qml.device("default.qubit", wires=N_QUBITS)
rng = np.random.default_rng(SEED)
RAND_W = rng.uniform(0, 2*np.pi, size=(QUANV_LAYERS, N_QUBITS))     # fixed random "kernel"

@qml.qnode(dev, interface="numpy")
def quanv_circuit(phi):                    # phi: (batch, 4) -> parameter broadcasting
    for w in range(N_QUBITS):
        qml.RY(np.pi * phi[:, w], wires=w)                          # encoding
    qml.RandomLayers(RAND_W, wires=range(N_QUBITS), seed=SEED)      # quantum convolution
    return [qml.expval(qml.PauliZ(w)) for w in range(N_QUBITS)]

def quanvolve(X_gray_u8, chunk_imgs=64):
    # X: (N,40,40,1) uint8 -> (N,20,20,4) float32
    N, H, W, _ = X_gray_u8.shape; h, w = H//2, W//2
    out = np.zeros((N, h, w, N_QUBITS), np.float32)
    for s in range(0, N, chunk_imgs):
        x = X_gray_u8[s:s+chunk_imgs, ..., 0].astype(np.float32) / 255.0
        n = len(x)
        patches = x.reshape(n, h, 2, w, 2).transpose(0, 1, 3, 2, 4).reshape(-1, 4)    # all 2x2 patches
        res = np.stack([np.asarray(r) for r in quanv_circuit(patches)], axis=-1)       # (n*h*w, 4)
        out[s:s+n] = res.reshape(n, h, w, N_QUBITS)
    return out

def cached_quanv(name, X):
    f = OUT_DIR / f"quanv_{name}.npy"
    if f.exists(): return np.load(f)
    t = time.time(); q = quanvolve(to_gray_size(X, QML_SIZE)); np.save(f, q)
    print(f"{name}: {q.shape} in {time.time()-t:.0f}s"); return q

Qtr, Qva, Qte = cached_quanv("train", Xtr), cached_quanv("val", Xva), cached_quanv("test", Xte)

fig, ax = plt.subplots(1, 5, figsize=(12, 2.6))
ax[0].imshow(to_gray_size(Xte[:1], QML_SIZE)[0, ..., 0], cmap="gray"); ax[0].set_title("input 40x40")
for k in range(4): ax[k+1].imshow(Qte[0, ..., k], cmap="viridis"); ax[k+1].set_title(f"quantum ch {k}")
for a in ax: a.axis("off")
plt.show()

## 5. Models: CML, QML and the four baselines

In [ ]:
def build_cml():
    # 1 Conv + 1 Flatten + 2 Dense (Section 3.3.1), input 150x150x3, 3x3 kernel, ReLU
    i = layers.Input((CML_SIZE, CML_SIZE, 3)); x = layers.Rescaling(1/255.)(i)
    x = layers.Conv2D(32, 3, activation="relu", name="conv_L")(x)       # number of filters not given in paper -> 32
    x = layers.Flatten()(x); x = layers.Dropout(DROPOUT_CML)(x)
    x = layers.Dense(64, activation="relu")(x)
    return keras.Model(i, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="CML")

def build_qml():
    # quantum image (20x20x4) -> Flatten -> Dense512 -> Dense64 -> Dropout -> Softmax (Section 3.3.2)
    i = layers.Input(Qtr.shape[1:])
    x = layers.Flatten()(i)
    x = layers.Dense(512, activation="relu")(x); x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(DROPOUT_QML)(x)
    return keras.Model(i, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="QML")

def build_baseline(name):
    app = {"VGG19": keras.applications.VGG19, "DenseNet201": keras.applications.DenseNet201,
           "ResNet152": keras.applications.ResNet152, "InceptionV3": keras.applications.InceptionV3}[name]
    i = layers.Input((CML_SIZE, CML_SIZE, 3)); x = layers.Rescaling(1/255.)(i)
    base = app(weights=BASELINE_WEIGHTS, include_top=False, input_shape=(CML_SIZE, CML_SIZE, 3), pooling="avg")
    x = base(x)
    return keras.Model(i, layers.Dense(NUM_CLASSES, activation="softmax")(x), name=name)

MODELS = {"VGG19": ("img", lambda: build_baseline("VGG19")), "DenseNet201": ("img", lambda: build_baseline("DenseNet201")),
          "ResNet152": ("img", lambda: build_baseline("ResNet152")), "InceptionV3": ("img", lambda: build_baseline("InceptionV3")),
          "CML": ("img", build_cml), "QML": ("quantum", build_qml)}
if QUICK_TEST: MODELS = {k: MODELS[k] for k in ["CML", "QML", "VGG19"]}
build_cml().summary(); build_qml().summary()

## 6. Performance metrics (Eqs. 24–31)
Multi-class → one-vs-rest from the confusion matrix, macro-averaged: **Specificity (P1), Sensitivity (P2), Precision (P3), Accuracy (P4), F1 (P5), MCC (P6)**.

In [ ]:
def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES)); tot = cm.sum()
    TP = np.diag(cm).astype(float); FP = cm.sum(0) - TP; FN = cm.sum(1) - TP; TN = tot - TP - FP - FN
    eps = 1e-12
    spec, sens, prec = TN/(TN+FP+eps), TP/(TP+FN+eps), TP/(TP+FP+eps)
    f1 = 2*prec*sens/(prec+sens+eps)
    return {"Specificity": spec.mean()*100, "Sensitivity": sens.mean()*100, "Precision": prec.mean()*100,
            "Accuracy": TP.sum()/tot*100, "F1": f1.mean()*100, "MCC": matthews_corrcoef(y_true, y_pred)*100}, cm

def get_data(kind): return ((Xtr, Xva, Xte) if kind == "img" else (Qtr, Qva, Qte))

## 7. Train / validate / test – every model, 5 independent runs

In [ ]:
rows, cms, best = [], {}, {}
for mname, (kind, builder) in MODELS.items():
    A_tr, A_va, A_te = get_data(kind)
    for run in range(1, N_RUNS+1):
        keras.backend.clear_session(); tf.random.set_seed(SEED+run); np.random.seed(SEED+run)
        model = builder()
        model.compile(optimizer=keras.optimizers.Adam(LR, beta_1=0.9), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        cb = [keras.callbacks.EarlyStopping("val_loss", patience=PATIENCE, restore_best_weights=True)] if EARLY_STOP else []
        t = time.time()
        model.fit(A_tr, ytr, validation_data=(A_va, yva), epochs=EPOCHS, batch_size=BATCH, callbacks=cb, verbose=0)
        pred = model.predict(A_te, batch_size=128, verbose=0).argmax(1)
        m, cm = compute_metrics(yte, pred)
        rows.append({"Model": mname, "Run": run, **m}); cms[(mname, run)] = cm
        print(f"{mname:12s} run {run}: acc {m['Accuracy']:.2f}  F1 {m['F1']:.2f}  MCC {m['MCC']:.2f}  ({time.time()-t:.0f}s)")
        if mname not in best or m["Accuracy"] > best[mname][0]:
            best[mname] = (m["Accuracy"], run); model.save(OUT_DIR/f"{mname}_best.keras")
res = pd.DataFrame(rows); res.to_csv(OUT_DIR/"all_runs.csv", index=False)

## 8. Results table (mean ± SD over runs) – compare with Table 5 of the paper

In [ ]:
metrics = ["Specificity", "Sensitivity", "Precision", "Accuracy", "F1", "MCC"]
g = res.groupby("Model", sort=False)[metrics]
summary = (g.mean().round(2).astype(str) + " ± " + g.std().round(2).astype(str))
summary.columns = ["P1 Spec", "P2 Sens", "P3 Prec", "P4 Acc", "P5 F1", "P6 MCC"]
summary.to_csv(OUT_DIR/"summary_mean_sd.csv"); display(summary)

res.groupby("Model", sort=False)[metrics].mean().plot.bar(figsize=(11, 4), ylim=(0, 100), rot=0)
plt.ylabel("%"); plt.title("Mean metrics over runs"); plt.legend(ncol=6, fontsize=8, loc="lower center"); plt.show()

## 9. Confusion matrices (best run) and Grad-CAM

In [ ]:
fig, ax = plt.subplots(1, len(best), figsize=(4.2*len(best), 4))
ax = np.atleast_1d(ax)
for a, (mname, (acc, run)) in zip(ax, best.items()):
    sns.heatmap(cms[(mname, run)], annot=True, fmt="d", cmap="Blues", cbar=False, ax=a,
                xticklabels=["LwD", "LwoD", "OSCC", "Norm"], yticklabels=["LwD", "LwoD", "OSCC", "Norm"])
    a.set_title(f"{mname} (run {run}, acc {acc:.1f}%)"); a.set_xlabel("Predicted"); a.set_ylabel("True")
plt.tight_layout(); plt.show()

In [ ]:
def grad_cam(model, x, layer_name, class_idx=None):
    # standard Grad-CAM for a (nested) conv layer; x: (1,H,W,C)
    gm = keras.Model(model.inputs, [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as t:
        fm, p = gm(x); c = int(tf.argmax(p[0])) if class_idx is None else class_idx; s = p[:, c]
    w = tf.reduce_mean(t.gradient(s, fm), axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * fm, -1))[0].numpy()
    return cam / (cam.max() + 1e-8), c

def saliency_quantum(model, q):
    # QML: the quantum image is the model input, so gradient x input w.r.t. it gives the heat-map
    q = tf.convert_to_tensor(q)
    with tf.GradientTape() as t:
        t.watch(q); p = model(q); c = int(tf.argmax(p[0])); s = p[:, c]
    cam = tf.reduce_mean(tf.abs(t.gradient(s, q) * q), -1)[0].numpy()
    return cam / (cam.max() + 1e-8), c

cml = keras.models.load_model(OUT_DIR/"CML_best.keras"); qml_m = keras.models.load_model(OUT_DIR/"QML_best.keras")
idx = [int(np.where(yte == c)[0][0]) for c in range(NUM_CLASSES)]            # one test image per class
fig, ax = plt.subplots(3, NUM_CLASSES, figsize=(3.2*NUM_CLASSES, 9))
for j, i in enumerate(idx):
    img = Xte[i]
    cam, pc = grad_cam(cml, img[None].astype("float32"), "conv_L")
    cam = tf.image.resize(cam[..., None], (CML_SIZE, CML_SIZE)).numpy()[..., 0]
    ax[0, j].imshow(img); ax[0, j].set_title(f"True: {CLASS_NAMES[yte[i]]}", fontsize=8)
    ax[1, j].imshow(img); ax[1, j].imshow(cam, cmap="jet", alpha=.45); ax[1, j].set_title(f"CML Grad-CAM -> {CLASS_NAMES[pc]}", fontsize=8)
    qc, qp = saliency_quantum(qml_m, Qte[i:i+1]); qc = tf.image.resize(qc[..., None], (CML_SIZE, CML_SIZE)).numpy()[..., 0]
    ax[2, j].imshow(img); ax[2, j].imshow(qc, cmap="jet", alpha=.45); ax[2, j].set_title(f"QML saliency -> {CLASS_NAMES[qp]}", fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(OUT_DIR/"gradcam.png", dpi=150); plt.show()